# Phase 6 — đo LLM dự phòng trên Colab (GPU)

LLM chỉ được gọi cho câu mà luật **bỏ trống**. Notebook đo:
1. độ đúng LLM trên các câu train luật bỏ trống (có gold) → quyết định bật/tắt LLM;
2. thời gian chạy trọn bộ public (giới hạn thi: 6 giờ);
3. tính tất định: chạy public 2 lần phải ra cùng kết quả (temperature 0, seed 42).

**Chuẩn bị:** Runtime → GPU (T4/L4/A100). Đặt `DATASET.zip` (bên trong có `train/` và `public_test/`, train có `answers.jsonl`) ở `MyDrive/DATAMATE/DATASET.zip`.
Kết quả lưu về `MyDrive/DATAMATE/phase6_out/` — gửi lại thư mục đó (hoặc chép bảng in ở ô cuối) để phân tích.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/DATAMATE'
!mkdir -p {DRIVE}/phase6_out
!rm -rf /content/DATASET && mkdir -p /content/DATASET && unzip -q {DRIVE}/DATASET.zip -d /content/DATASET
!ls /content/DATASET /content/DATASET/train | head

In [ ]:
%cd /content
!rm -rf DataMate && git clone -q https://github.com/LingSeanCoder/DataMate.git
%cd /content/DataMate
!git log --oneline -1
!pip -q install -r requirements.txt huggingface_hub

### Cài llama.cpp server (CUDA)
Cách A: wheel dựng sẵn của `llama-cpp-python` (nhanh). Nếu lỗi, dùng cách B (build llama.cpp từ nguồn, ~10 phút) ở ô sau.

In [ ]:
!pip -q install "llama-cpp-python[server]" --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!python -c "import llama_cpp; print('llama_cpp', llama_cpp.__version__)"

In [ ]:
# Cách B (chỉ chạy nếu cách A lỗi):
# !git clone -q https://github.com/ggml-org/llama.cpp /content/llama.cpp && cd /content/llama.cpp && cmake -B build -DGGML_CUDA=ON && cmake --build build -j --target llama-server
USE_LLAMA_CPP_BINARY = False

In [ ]:
from huggingface_hub import snapshot_download
import glob
p = snapshot_download('Qwen/Qwen2.5-Coder-14B-Instruct-GGUF', allow_patterns=['*q4_k_m*.gguf'], local_dir='/content/model')
MODEL = sorted(glob.glob('/content/model/*q4_k_m*.gguf'))[0]  # file nhiều phần: đưa phần đầu, llama.cpp tự nạp phần sau
print(MODEL)

In [ ]:
import subprocess, time, urllib.request, os
if USE_LLAMA_CPP_BINARY:
    cmd = f"/content/llama.cpp/build/bin/llama-server -m {MODEL} -ngl 999 -c 16384 --port 8000 --alias Qwen2.5-Coder-14B-Instruct"
else:
    cmd = (f"python -m llama_cpp.server --model {MODEL} --n_gpu_layers -1 --n_ctx 16384 --port 8000 "
           f"--chat_format chatml --model_alias Qwen2.5-Coder-14B-Instruct")
server = subprocess.Popen(cmd + " > /content/server.log 2>&1", shell=True)
for _ in range(180):
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=2); print('server sẵn sàng'); break
    except Exception:
        time.sleep(5)
else:
    print(open('/content/server.log').read()[-3000:])
os.environ['LOCAL_LLM_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['LOCAL_LLM_MODEL'] = 'Qwen2.5-Coder-14B-Instruct'

### Build DB + chạy chỉ-luật (mốc so sánh)

In [ ]:
import time
T = {}
for name, root in (('train', '/content/DATASET/train'), ('public', '/content/DATASET/public_test')):
    t = time.time()
    !python -W ignore main.py --data-root {root}/data --questions {root}/questions.jsonl --db-dir db_{name} --rebuild --no-llm \
        --out out_{name}_rules.jsonl --log log_{name}_rules.jsonl --overwrite --workers 4 > /dev/null
    T[f'{name}_build+rules'] = round(time.time() - t)
print(T)
!python eval.py --sub out_train_rules.jsonl --answers /content/DATASET/train/answers.jsonl --questions /content/DATASET/train/questions.jsonl --wrong 0 | tail -1

### Chạy có LLM (câu luật bỏ trống mới gọi LLM)

In [ ]:
for name, root, tag in (('train', '/content/DATASET/train', 'llm'), ('public', '/content/DATASET/public_test', 'llm1'),
                        ('public', '/content/DATASET/public_test', 'llm2')):
    t = time.time()
    !python -W ignore main.py --data-root {root}/data --questions {root}/questions.jsonl --db-dir db_{name} --skip-build \
        --out out_{name}_{tag}.jsonl --log log_{name}_{tag}.jsonl --overwrite --workers 2 > /dev/null
    T[f'{name}_{tag}'] = round(time.time() - t)
print(T)
!python eval.py --sub out_train_llm.jsonl --answers /content/DATASET/train/answers.jsonl --questions /content/DATASET/train/questions.jsonl --wrong 0 | tail -1

### Báo cáo

In [ ]:
import json, sys
sys.path.insert(0, '.')
from eval import score_one
load = lambda p: {json.loads(l)['id']: json.loads(l) for l in open(p)}
gold = load('/content/DATASET/train/answers.jsonl')
logs = [json.loads(l) for l in open('log_train_llm.jsonl')]
llm = [r for r in logs if r.get('route') == 'llm']
s = [score_one(r['answer'], gold[r['id']]) for r in llm]
print(f"TRAIN: {len(llm)} câu đi LLM | đúng {s.count(1)}  sai {s.count(-1)}  trống {s.count(0)}  => điểm LLM thêm {sum(s)}")
for r, v in zip(llm, s):
    print(f"  [{v:+d}] {r['question'][:80]} | LLM={json.dumps(r['answer'].get('answer'), ensure_ascii=False)[:40]} | gold={json.dumps(gold[r['id']].get('answer'), ensure_ascii=False)[:40]} | sql={(r.get('final_sql') or '')[:120]}")
a, b = load('out_public_llm1.jsonl'), load('out_public_llm2.jsonl')
diff = [i for i in a if a[i] != b[i]]
plog = [json.loads(l) for l in open('log_public_llm1.jsonl')]
print(f"PUBLIC: {sum(r.get('route') == 'llm' for r in plog)} câu đi LLM | 2 lần chạy khác nhau ở {len(diff)} câu | thời gian (giây): {T}")
for r in plog:
    if r.get('route') == 'llm':
        print(f"  {r['question'][:80]} | LLM={json.dumps(r['answer'].get('answer'), ensure_ascii=False)[:40]} | sql={(r.get('final_sql') or '')[:120]}")
!cp out_*.jsonl log_*.jsonl /content/server.log {DRIVE}/phase6_out/
print('đã lưu vào', DRIVE + '/phase6_out')